In [2]:
import pandas as pd
from pathlib import Path
import os
import numpy as np
from geopy.distance import geodesic
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from lifelines import WeibullFitter
import math
import numpy as np
import pandas as pd
import folium
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
import datetime
from pathlib import Path
import os
import pandas as pd
from tabulate import tabulate


In [3]:

# ----------------------------
# Directories setup
# ----------------------------
notebook_dir = Path(os.getcwd())
trip_base_dir = notebook_dir / "trip_data" / "raw_files"
raw_maintenance_files_dir = notebook_dir / "maintenance_data" / "raw_files"
_maintenance_files = sorted(raw_maintenance_files_dir.glob("*.csv"))


# ----------------------------
# Single Clean Summary
# ----------------------------
final_summary = [
    ["Maintenance files", len(_maintenance_files)]
]

print("\nLOAD COMPLETE")
print(tabulate(final_summary, headers=["Metric", "Value"], tablefmt="github"))


LOAD COMPLETE
| Metric            |   Value |
|-------------------|---------|
| Maintenance files |       3 |


In [4]:
# ---------------------------------------------------------
# Load Maintenance Data
# ---------------------------------------------------------
maint_store = {'damage': [], 'repair': [], 'maintenance': []}

for file in _maintenance_files:
    df = pd.read_csv(file)
    fname = file.name.lower()
    
    # Sort into correct bucket based on filename
    category = 'damage' if 'damage' in fname else 'repair' if 'repair' in fname else 'maintenance'
    maint_store[category].append(df)

# ---------------------------------------------------------
# Create DataFrames dynamically
# ---------------------------------------------------------
damage_df = pd.concat(maint_store['damage'], ignore_index=True) if maint_store['damage'] else pd.DataFrame()
repair_df = pd.concat(maint_store['repair'], ignore_index=True) if maint_store['repair'] else pd.DataFrame()
maintenance_df = pd.concat(maint_store['maintenance'], ignore_index=True) if maint_store['maintenance'] else pd.DataFrame()

# ---------------------------------------------------------
# Concise, pretty summary
# ---------------------------------------------------------
stats = [
    ["Damage Logs", f"{len(damage_df):,}"],
    ["Repair Logs", f"{len(repair_df):,}"],
    ["Maint Logs",  f"{len(maintenance_df):,}"]
]

print("\n" + "="*30)
print(tabulate(stats, headers=["Dataset", "Rows"], tablefmt="github"))
print("="*30)


| Dataset     | Rows    |
|-------------|---------|
| Damage Logs | 756,565 |
| Repair Logs | 250,815 |
| Maint Logs  | 363,324 |


In [5]:
category_map = {
    # --- MECHANICAL / WEAR BASED ---
    'Bremse(r)': 'Braking System', 
    'Brake(s) need adjustment': 'Braking System',
    
    'Gir': 'Drivetrain', 'Belte / Belt': 'Drivetrain', 
    'Pedaler': 'Drivetrain', 'Cranck bearing/bottom bracket ': 'Drivetrain',
    
    'Hjul': 'Wheels & Tires', 'Lite luft': 'Wheels & Tires',
    
    'Styre': 'Steering & Chassis', 'Styrelager': 'Steering & Chassis', 'Frame': 'Steering & Chassis',
    
    'Sete': 'Body & Accessories', 'Setepinneklemme': 'Body & Accessories', 
    'Støtte': 'Body & Accessories', 'Skjerm(er)': 'Body & Accessories', 
    'Ringeklokke': 'Body & Accessories', 'Basket': 'Body & Accessories',
    
    # --- ELECTRONIC/SYSTEM ---
     'Console': 'Electronics', 
    'GPS': 'Electronics', 'Battery': 'Electronics', 'Lys': 'Electronics',
    
    'Lock & unlock': 'Locking System', 'Lås': 'Locking System',
    
    # --- LOGIC & EXTERNAL (Exclude from odometer analysis) ---
    'Too many quick returns': 'Too many quick returns',
    'Unauthorized Trip': 'Unauthorized Trip',
    'Unresponsive Controller': 'Unresponsive Controller',
    'Vandalism': 'Vandalism'
}

In [6]:
# apply mapping to create new column
damage_df['damage_category'] = damage_df['damage_type_name'].map(category_map).fillna('Other')

In [7]:
# get how many bike_set_unavailable within each category of damage
damage_counts = damage_df['damage_category'].value_counts().reset_index()
damage_counts.columns = ['Damage Category', 'Count']
print("\nDamage Category Counts:")
print(tabulate(damage_counts, headers="keys", tablefmt="github"))


# records that are damage category of "Too many quick returns", "Unauthorized Trip", "Unresponsive Controller", "Vandalism" are likely not related to odometer readings, so we can exclude them from the analysis of odometer readings and bike unavailability
damage_df = damage_df[~damage_df['damage_category'].isin(['Too many quick returns', 'Unauthorized Trip', 'Unresponsive Controller', 'Vandalism', 'Other'])].copy()


# get percentage of total dmages within each category that are bike_set_unavailable
damage_df['is_unavailable'] = damage_df['set_vehicle_unavailable'].astype(bool)
unavailable_percentages = damage_df.groupby('damage_category')['is_unavailable'].mean().reset_index()
unavailable_percentages.columns = ['Damage Category', 'Percent Unavailable']
print("\nPercentage of Unavailable Bikes by Damage Category:")
print(tabulate(unavailable_percentages, headers="keys", tablefmt="github"))



Damage Category Counts:
|    | Damage Category         |   Count |
|----|-------------------------|---------|
|  0 | Other                   |  341987 |
|  1 | Unresponsive Controller |  173447 |
|  2 | Too many quick returns  |   64051 |
|  3 | Drivetrain              |   55468 |
|  4 | Wheels & Tires          |   33239 |
|  5 | Unauthorized Trip       |   28213 |
|  6 | Braking System          |   16424 |
|  7 | Steering & Chassis      |   16400 |
|  8 | Body & Accessories      |   11595 |
|  9 | Locking System          |    9550 |
| 10 | Electronics             |    5944 |
| 11 | Vandalism               |     247 |

Percentage of Unavailable Bikes by Damage Category:
|    | Damage Category    |   Percent Unavailable |
|----|--------------------|-----------------------|
|  0 | Body & Accessories |             0.0450194 |
|  1 | Braking System     |             0.958171  |
|  2 | Drivetrain         |             0.37582   |
|  3 | Electronics        |             0.787853  |
|  4 | L

In [18]:
import pandas as pd
from tabulate import tabulate

maintenance_key = "asset_maintenance_id"  # change if needed

merged_df = damage_df.merge(
    maintenance_df,
    left_on="asset_maintenance_id",
    right_on=maintenance_key,
    how="left",
    suffixes=("", "_maint")
)

print("\nMerged DataFrame Columns:")
print(merged_df.columns.tolist())

# Datetimes
merged_df["started_at"] = pd.to_datetime(merged_df["started_at"], errors="coerce")
merged_df["completed_at"] = pd.to_datetime(merged_df["completed_at"], errors="coerce")

# Filter
filtered_df = merged_df[
    (merged_df["set_vehicle_unavailable"] == False) &
    merged_df["started_at"].notna() &
    merged_df["completed_at"].notna()
].copy()

# Duration (minutes)
filtered_df["maintenance_minutes"] = (
    filtered_df["completed_at"] - filtered_df["started_at"]
).dt.total_seconds() / 60

# Remove negative durations (and optionally zero)
filtered_df = filtered_df[filtered_df["maintenance_minutes"] > 0]

# Count damage rows per maintenance record (duplicates included by design)
filtered_df["damage_count_in_maintenance"] = (
    filtered_df.groupby("asset_maintenance_id")["asset_maintenance_id"].transform("size")
)

# Allocate equal share per damage row
filtered_df["allocated_repair_minutes"] = (
    filtered_df["maintenance_minutes"] / filtered_df["damage_count_in_maintenance"]
)

# Average allocated repair time by damage category
avg_alloc = (
    filtered_df.groupby("damage_category")["allocated_repair_minutes"]
    .mean()
    .reset_index()
)
avg_alloc.columns = ["Damage Category", "Avg Allocated Repair Time (minutes)"]

print("\nAverage Allocated Repair Time by Damage Category (NOT set unavailable):")
print(tabulate(avg_alloc, headers="keys", tablefmt="github"))

# Percent of damages with allocated repair time > 10 minutes
filtered_df["is_greater_than_10"] = filtered_df["allocated_repair_minutes"] > 5

percentage_gt_5 = (
    filtered_df.groupby("damage_category")["is_greater_than_10"]
    .mean()
    .mul(100)
    .reset_index()
)
percentage_gt_5.columns = ["Damage Category", "Percent with Allocated Repair Time > 5 min"]

print("\nPercentage of Damages with Allocated Repair Time > 5 minutes by Damage Category:")
print(tabulate(percentage_gt_5, headers="keys", tablefmt="github"))

# Examples
examples_gt_5 = filtered_df.loc[filtered_df["is_greater_than_10"],
    ["asset_maintenance_id", "damage_category", "allocated_repair_minutes", "maintenance_minutes", "damage_count_in_maintenance"]
].head(10)

print("\nExamples of Damages with Allocated Repair Time > 5 minutes:")
print(tabulate(examples_gt_5, headers="keys", tablefmt="github"))


Merged DataFrame Columns:
['vehicle_id', 'asset_model_id', 'asset_model_name', 'vehicle_category', 'case', 'damage_id', 'created_at', 'comment', 'damage_type_id', 'damage_type_name', 'set_vehicle_unavailable', 'reported_by_administrator_id', 'reported_by_user_id', 'repeats', 'resolved_at', 'asset_maintenance_id', 'damage_category', 'is_unavailable', 'repair_time_hours', 'vehicle_id_maint', 'asset_model_id_maint', 'asset_model_name_maint', 'vehicle_category_maint', 'case_maint', 'started_at', 'completed_at', 'comment_maint', 'completed_by_administrator_id']

Average Allocated Repair Time by Damage Category (NOT set unavailable):
|    | Damage Category    |   Avg Allocated Repair Time (minutes) |
|----|--------------------|---------------------------------------|
|  0 | Body & Accessories |                              41.7024  |
|  1 | Braking System     |                               6.87474 |
|  2 | Drivetrain         |                              55.0338  |
|  3 | Electronics     